# Silver Layer Transformation Pipeline - Generic & Metadata-Driven

Generic notebook to apply transformations, deduplicate, and merge Bronze data into Silver Delta tables.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
dbutils.widgets.text("source_table", "customers", "Source Bronze Table")
dbutils.widgets.text("target_table", "customers_clean", "Target Silver Table")
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "staging", "prod"], "Environment / Stage")
dbutils.widgets.dropdown("use_scd_type2", "false", ["true", "false"], "Use SCD Type 2")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.dropdown("create_views", "false", ["false", "true"], "Create Views (Optional)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
try:
    _stage = dbutils.widgets.get("stage").strip()
except Exception:
    _stage = ""
environment = _stage or dbutils.widgets.get("environment")
use_scd_type2 = dbutils.widgets.get("use_scd_type2").lower() == "true"
client_id = dbutils.widgets.get("client_id").strip() or None
create_views = dbutils.widgets.get("create_views").lower() == "true"

print(f"Executing Silver transformations from {source_table} to {target_table} (client: {client_id or 'global'}, SCD2: {use_scd_type2})")


In [ ]:
# Imports & Path setup
import sys

from lakeforge import (
    ConfigParser,
    SilverTransformer,
    SilverMergeEngine,
    create_scd_type2_handler,
    create_trust_engine
)

print("✅ Framework libraries loaded")

In [ ]:
# Load Configurations
env_config = ConfigParser.get_environment_config(config_dir, environment)

try:
    ing_config = ConfigParser.get_ingestion_config(config_dir, source_table, environment)
    bronze_catalog = ing_config.target_catalog
    bronze_schema = ing_config.target_schema
    silver_catalog = ing_config.target_catalog
    silver_schema = ing_config.silver_schema or env_config["silver_schema"]
except Exception:
    bronze_catalog = env_config["catalog"]
    bronze_schema = env_config["bronze_schema"]
    silver_catalog = env_config["catalog"]
    silver_schema = env_config["silver_schema"]

full_source_table = f"{bronze_catalog}.{bronze_schema}.{source_table}"
full_target_table = f"{silver_catalog}.{silver_schema}.{target_table}"

print(f"Source: {full_source_table}")
print(f"Target: {full_target_table}")


In [ ]:
# Read Bronze Table
df_bronze = spark.table(full_source_table)
print(f"✅ Loaded Bronze table: {df_bronze.count()} records")

In [ ]:
# Load Configurations
env_config = ConfigParser.get_environment_config(config_dir, environment)
catalog = env_config["catalog"]
bronze_schema = env_config["bronze_schema"]
silver_schema = env_config["silver_schema"]

trans_config = ConfigParser.get_silver_transformation_config(config_dir, target_table, environment, client_id=client_id)
print(f"Parsed Silver Config for: {target_table}")
print(trans_config)


In [ ]:
# Write / Merge to Silver layer
ing_config = ConfigParser.get_ingestion_config(config_dir, source_table, environment)
business_keys = ing_config.business_key
merge_strat = ing_config.merge_strategy

if use_scd_type2:
    print("ℹ️ Performing SCD Type 2 merge...")
    scd_handler = create_scd_type2_handler(spark)
    # Remove existing SCD columns to avoid double calculation
    clean_cols = [c for c in df_silver.columns if c not in ["effective_start_date", "effective_end_date", "is_current", "record_hash"]]
    df_silver_clean = df_silver.select(*clean_cols)
    
    merge_stats = scd_handler.merge_scd_type2(
        source_df=df_silver_clean,
        target_table=target_table,
        catalog=silver_catalog,
        schema=silver_schema,
        business_keys=business_keys
    )
else:
    print(f"ℹ️ Performing standard merge/write ({merge_strat})...")
    mode_val = "merge" if merge_strat == "upsert" else "append" if merge_strat == "append" else "overwrite"
    merge_stats = SilverMergeEngine.merge(
        spark=spark,
        df=df_silver,
        target_table=full_target_table,
        merge_keys=business_keys,
        mode=mode_val
    )

print(f"✅ Write complete: {merge_stats}")

In [ ]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)
trust_validations = [
    {
        "type": "row_count",
        "params": {
            "source_df": df_bronze,
            "target_df": df_silver,
            "tolerance_percent": 5.0
        }
    },
    {
        "type": "duplicate_explosion",
        "params": {
            "source_df": df_bronze,
            "target_df": df_silver,
            "key_columns": business_keys,
            "max_explosion_ratio": 1.5
        }
    }
]

trust_results = trust_engine.run_trust_validations(trust_validations)
trust_score = trust_engine.calculate_trust_score(
    table_name=full_target_table,
    dq_results=trust_results,
    pipeline_stage="silver"
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")

In [ ]:
# Create Views if explicitly enabled (otherwise run VIEWS_Creation_Pipeline on-demand)
if create_views:
    try:
        print("Checking for configured views to create...")
        from lakeforge.views.view_manager import create_view_manager
        from lakeforge.metadata.config_parser import load_view_configs
        env_config = ConfigParser.get_environment_config(config_dir, environment)
        view_configs = load_view_configs(config_dir, environment, client_id=client_id)
        target_views = [vc for vc in view_configs if vc.schema == env_config["silver_schema"]]
        if target_views:
            view_manager = create_view_manager(spark, env_config)
            view_manager.create_views(target_views)
            print(f"Created {len(target_views)} views successfully")
        else:
            print("No views configured for silver schema")
    except Exception as e:
        print(f"Error creating views: {str(e)}")
else:
    print("View creation skipped (decoupled; trigger VIEWS_Creation_Pipeline on demand or set create_views=true)")
